<a href="https://colab.research.google.com/github/nttqyn/Project-III/blob/main/FedGAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
mkdir -p data/raw

In [3]:
import torch
import os

print(f"Torch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

# 1. Cài đặt các thư viện phụ thuộc của PyTorch Geometric (Quan trọng: phải khớp version)
# Lệnh này tìm file wheel (.whl) tương ứng để không phải build lại
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html

# 2. Cài đặt Torch Geometric
!pip install torch-geometric>=2.0.0

# 3. Cài đặt các thư viện còn lại (Thường Colab đã có sẵn, nhưng lệnh này đảm bảo đúng version)
!pip install scikit-learn>=1.0.0 pandas>=1.3.0 numpy>=1.21.0 matplotlib>=3.5.0

print("=== Đã cài đặt xong môi trường ===")


Torch version: 2.8.0+cu126
CUDA version: 12.6
Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 107.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 70.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 76.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.4 MB/s eta 0:00:00
=== Đã cài đặt xong môi trường ===


In [4]:
%%writefile config.py
import torch
import os

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if '__file__' in locals() else os.getcwd()

class Config:
    # Data directories
    DATA_RAW_PATH = os.path.join(BASE_DIR, 'data', 'raw')
    DATA_PROCESSED_PATH = os.path.join(BASE_DIR, 'data', 'processed')
    MODELS_PATH = os.path.join(BASE_DIR, 'models')
    RESULTS_PATH = os.path.join(BASE_DIR, 'results')

    # Dataset files
    TRAIN_20P_FILE = 'KDDTrain+_20Percent.txt'
    TRAIN_FULL_FILE = 'KDDTrain+.txt'
    TEST_FILE = 'KDDTest+.txt'
    TEST_21_FILE = 'KDDTest-21.txt'

    # Model parameters
    IN_FEATURES = 42  # 41 features + 1 log density
    HIDDEN_FEATURES = 64
    OUT_FEATURES = 5  # 5 classes: Normal, DoS, Probe, R2L, U2R
    HEADS = 8
    DROPOUT = 0.5

    #GNN parameters
    GNN_LAYERS = 5

    # Federated Learning parameters
    NUM_CLIENTS = 10
    COMMUNICATION_ROUNDS = 50
    LOCAL_EPOCHS = 10
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 5e-4

    # Training parameters
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    SEED = 42
    TIME_WINDOW = 1
    BATCH_SIZE = 128
    CLASS_WEIGHTS = None

    # Data preprocessing
    CATEGORICAL_FEATURES = ['protocol_type', 'service', 'flag']
    NUMERICAL_FEATURES = [
        'duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
        'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
        'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
        'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
        'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
        'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
        'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
        'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
        'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
        'dst_host_serror_rate', 'dst_host_srv_serror_rate',
        'dst_host_rerror_rate', 'dst_host_srv_rerror_rate'
    ]

    # Attack type mapping for 5-class classification
    ATTACK_MAPPING = {
        'normal': 0,
        'dos': 1,
        'probe': 2,
        'r2l': 3,
        'u2r': 4
    }

    # Specific attack subtypes mapping to main categories
    ATTACK_CATEGORIES = {
        'dos': ['back', 'land', 'neptune', 'pod', 'smurf', 'teardrop', 'apache2', 'udpstorm', 'processtable', 'worm'],
        'probe': ['satan', 'ipsweep', 'nmap', 'portsweep', 'mscan', 'saint'],
        'r2l': ['guess_passwd', 'ftp_write', 'imap', 'phf', 'multihop', 'warezmaster', 'warezclient', 'spy', 'xlock', 'xsnoop', 'snmpguess', 'snmpgetattack', 'httptunnel', 'sendmail', 'named'],
        'u2r': ['buffer_overflow', 'loadmodule', 'rootkit', 'perl', 'sqlattack', 'xterm', 'ps']
    }

config = Config()

Writing config.py


In [5]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.inits import glorot
from torch_geometric.utils import softmax
from config import config

class CustomFedGATLayer(MessagePassing):
    """
    Triển khai tùy chỉnh của GAT layer dựa trên Công thức (8) của bài báo.
    """
    def __init__(self, in_features, out_features, heads=8, dropout=0.6, **kwargs):
        super().__init__(node_dim=0, aggr='add', **kwargs)

        self.in_features = in_features
        self.out_features = out_features
        self.heads = heads
        self.dropout = dropout

        self.W = nn.Linear(in_features, out_features * heads, bias=False)
        self.att_content = nn.Parameter(torch.Tensor(1, heads, 2 * out_features))

        # === SỬA LỖI RUNTIMEERROR TẠI ĐÂY ===
        # Shape cũ (lỗi): [1, heads, 1]
        # Shape mới (đúng): [1, heads]
        self.w_s_param = nn.Parameter(torch.Tensor(1, heads))

        self.dropout_layer = nn.Dropout(p=dropout)

        self.reset_parameters()

    def reset_parameters(self):
        glorot(self.W.weight)
        glorot(self.att_content)
        glorot(self.w_s_param) # Khởi tạo trọng số cấu trúc

    def forward(self, x, edge_index, structure_coeffs):
        x_transformed = self.W(x)
        x_transformed = x_transformed.view(-1, self.heads, self.out_features)

        num_nodes = x_transformed.size(0)

        out = self.propagate(edge_index,
                              size=(num_nodes, num_nodes),
                              x=x_transformed,
                              structure_coeffs=structure_coeffs)

        return out

    def message(self, x_i, x_j, index, ptr, size_i, structure_coeffs):
        """
        x_i: [E, heads, out_features] (Node đích)
        x_j: [E, heads, out_features] (Node nguồn)
        size_i: Số lượng nút đích (N)
        """

        # === A. TÍNH CONTENT ATTENTION (e_ij) ===
        x_cat = torch.cat([x_i, x_j], dim=-1)
        e_ij = F.leaky_relu((self.att_content * x_cat).sum(dim=-1), negative_slope=0.2)
        alpha_content = softmax(e_ij, index, ptr, num_nodes=size_i)

        # === B. TÍNH STRUCTURE ATTENTION (s_ij) ===
        s_ij = structure_coeffs.view(-1, 1).repeat(1, self.heads)
        alpha_structure = softmax(s_ij, index, ptr, num_nodes=size_i)

        # === C. KẾT HỢP (a_ij) - Công thức (8) ===

        # w_s_param giờ có shape [1, heads], khớp với alpha_content
        w_s = torch.sigmoid(self.w_s_param)
        w_e = 1.0 - w_s

        # Phép nhân này (broadcast) giờ đã ĐÚNG:
        # [1, 8] * [75574, 8]
        alpha = (w_e * alpha_content) + (w_s * alpha_structure)

        alpha = self.dropout_layer(alpha)

        # (Eq. 9: a_ij * Wh_j)
        return x_j * alpha.view(-1, self.heads, 1)

    def aggregate(self, inputs, index, dim_size):
        return super().aggregate(inputs, index, dim_size=dim_size)

    def update(self, aggr_out):
        return aggr_out.view(-1, self.heads * self.out_features)


# === LỚP MODEL CHÍNH (GATWithAttention) ===
class GATWithAttention(nn.Module):
    def __init__(self, in_features, hidden_features, out_features=5, heads=8, dropout=0.6):
        super(GATWithAttention, self).__init__()

        self.dropout_val = dropout
        self.num_layers = config.GNN_LAYERS

        self.layers = nn.ModuleList()

        # Layer 1: Input -> Hidden
        self.layers.append(CustomFedGATLayer(
            in_features,
            hidden_features,
            heads=heads,
            dropout=dropout
        ))

        # Layers 2 -> 22: Hidden -> Hidden (Deep layers)
        # Input của layer sau là (hidden_features * heads) từ layer trước
        # Output của layer này cũng là hidden_features (nhưng qua multi-head sẽ nhân lên)
        for _ in range(self.num_layers - 2):
            self.layers.append(CustomFedGATLayer(
                hidden_features * heads,
                hidden_features,
                heads=heads,
                dropout=dropout
            ))

        # Layer 23: Hidden -> Output
        self.layers.append(CustomFedGATLayer(
            hidden_features * heads,
            out_features,
            heads=1,
            dropout=dropout
        ))

    def forward(self, x, edge_index, structure_coeffs):
        # Forward qua các lớp ẩn
        for i in range(self.num_layers - 1):
            x_in = x # Lưu input để làm Residual Connection (nếu kích thước khớp)

            x = self.layers[i](x, edge_index, structure_coeffs)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout_val, training=self.training)

            # Residual Connection cho các lớp giữa
            if i > 0 and x.shape == x_in.shape:
                x = x + x_in

        # Output layer
        x = self.layers[-1](x, edge_index, structure_coeffs)

        return F.log_softmax(x, dim=1)

Writing model.py


In [6]:
%%writefile data_loader.py
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

from torch_geometric.utils import add_remaining_self_loops, get_laplacian
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os
from config import config
from sklearn.utils import resample

def load_nsl_kdd_data(file_name):
    """Load NSL-KDD dataset from raw text file"""
    file_path = os.path.join(config.DATA_RAW_PATH, file_name)

    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        alternative_path = os.path.join('data', 'raw', file_name)
        if os.path.exists(alternative_path):
            print(f"Found file at alternative path: {alternative_path}")
            file_path = alternative_path
        else:
            print(f"File also not found at: {alternative_path}")
            return None

    columns = [
        'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
        'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins',
        'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root',
        'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
        'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate',
        'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
        'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
        'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
        'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
        'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack_type', 'difficulty_level'
    ]

    try:
        df = pd.read_csv(file_path, header=None, names=columns)
        print(f"Successfully loaded {file_name} with {len(df)} records")
        return df
    except Exception as e:
        print(f"Error loading {file_name}: {e}")
        return None

def balance_dataset(df):
    """
    Cân bằng dữ liệu bằng cách nhân bản (Oversampling) các lớp thiểu số.
    Mục tiêu: Đưa số lượng mẫu các lớp hiếm lên mức chấp nhận được so với lớp đa số.
    """
    print("Balancing dataset (Oversampling rare classes)...")

    # Tách các lớp
    df_normal = df[df['label'] == 0]
    df_dos = df[df['label'] == 1]
    df_probe = df[df['label'] == 2]
    df_r2l = df[df['label'] == 3]
    df_u2r = df[df['label'] == 4]

    # Lấy số lượng mẫu của lớp DoS (thường là lớp nhiều thứ 2 sau Normal hoặc nhiều nhất) làm mốc
    # Hoặc lấy trung bình cộng. Ở đây ta lấy số lượng DoS để upscale các lớp kia lên.
    target_count = len(df_dos)

    # Hàm upscale
    def upscale(data, n_samples):
        if len(data) == 0: return data
        return resample(data, replace=True, n_samples=n_samples, random_state=42)

    # Upscale các lớp hiếm (Probe, R2L, U2R)
    # Lưu ý: U2R rất ít, nhân bản quá nhiều có thể gây Overfitting,
    # nhưng để cân bằng theo yêu cầu bài toán ta sẽ tăng nó lên.

    df_probe_upsampled = upscale(df_probe, target_count // 2) # Tăng Probe
    df_r2l_upsampled = upscale(df_r2l, target_count // 2)     # Tăng R2L
    df_u2r_upsampled = upscale(df_u2r, target_count // 4)     # Tăng U2R (ít hơn chút để tránh nhiễu quá lớn)

    # Ghép lại (Giữ nguyên Normal và DoS)
    df_balanced = pd.concat([
        df_normal,
        df_dos,
        df_probe_upsampled,
        df_r2l_upsampled,
        df_u2r_upsampled
    ])

    # Shuffle lại dữ liệu
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    print("Class distribution after balancing:")
    print(df_balanced['label'].value_counts())

    return df_balanced

def map_attack_type(attack_name):
    """Map specific attack names to 5 main categories"""
    attack_name = attack_name.lower()

    if attack_name == 'normal':
        return config.ATTACK_MAPPING['normal']

    for category, subtypes in config.ATTACK_CATEGORIES.items():
        if attack_name in subtypes:
            return config.ATTACK_MAPPING[category]

    # Default mapping
    if 'dos' in attack_name: return config.ATTACK_MAPPING['dos']
    elif 'probe' in attack_name: return config.ATTACK_MAPPING['probe']
    elif 'r2l' in attack_name: return config.ATTACK_MAPPING['r2l']
    elif 'u2r' in attack_name: return config.ATTACK_MAPPING['u2r']
    else: return config.ATTACK_MAPPING['dos']


def preprocess_train_data(df):
    """(1) CHỈ fit_transform dữ liệu train"""
    df = df.drop('difficulty_level', axis=1, errors='ignore')
    label_encoders = {}
    for col in config.CATEGORICAL_FEATURES:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le

    numerical_cols = [col for col in config.NUMERICAL_FEATURES if col in df.columns]
    scaler = StandardScaler()
    df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

    df['label'] = df['attack_type'].apply(map_attack_type)
    df = df.drop('attack_type', axis=1, errors='ignore')
    return df, label_encoders, scaler

def preprocess_test_data(df, label_encoders, scaler):
    """(2) CHỈ transform dữ liệu test DÙNG CHUNG encoder/scaler"""
    df = df.drop('difficulty_level', axis=1, errors='ignore')

    for col, le in label_encoders.items():
        if col in df.columns:
            df[col] = df[col].astype(str).apply(lambda x: x if x in le.classes_ else le.classes_[0])
            df[col] = le.transform(df[col])

    numerical_cols = [col for col in config.NUMERICAL_FEATURES if col in df.columns]
    if numerical_cols:
        df[numerical_cols] = scaler.transform(df[numerical_cols])

    df['label'] = df['attack_type'].apply(map_attack_type)
    df = df.drop('attack_type', axis=1, errors='ignore')
    return df

def build_temporal_graph(df, time_window=1): # Đảm bảo time_window=1
    """Build temporal graph (Sửa logic gán nhãn node VÀ TÍNH BẬC)"""
    if time_window != 1:
        print(f"Cảnh báo: TIME_WINDOW được đặt là {time_window}. Đề nghị đặt là 1 để có kết quả tốt nhất.")

    df = df.reset_index().rename(columns={'index': 'timestamp'})
    df['time_bucket'] = df['timestamp'] // time_window
    density = df.groupby('time_bucket').size().reset_index(name='log_density')

    node_features = []
    node_labels = []
    time_buckets = []

    feature_cols = [col for col in config.NUMERICAL_FEATURES + config.CATEGORICAL_FEATURES if col in df.columns]

    for bucket in density['time_bucket'].unique():
        bucket_data = df[df['time_bucket'] == bucket]

        if len(bucket_data) > 0:
            node_feature = bucket_data[feature_cols].mean().values
            log_density_val = density[density['time_bucket'] == bucket]['log_density'].values[0]
            node_feature = np.append(node_feature, log_density_val)

            # Sửa logic gán nhãn (5 lớp)
            node_labels_in_bucket = bucket_data['label']
            non_normal_labels = node_labels_in_bucket[node_labels_in_bucket != 0]

            if len(non_normal_labels) == 0:
                node_label = 0 # 0 = normal
            else:
                node_label = non_normal_labels.mode().iloc[0]

            node_features.append(node_feature)
            node_labels.append(node_label)
            time_buckets.append(bucket)

    if len(node_features) == 0:
        raise ValueError("No valid nodes created from the data")

    node_features = np.array(node_features)
    node_labels = np.array(node_labels)

    edge_index = []
    sorted_buckets = sorted(time_buckets)
    bucket_to_index = {bucket: idx for idx, bucket in enumerate(sorted_buckets)}

    for i in range(len(sorted_buckets) - 1):
        idx1 = bucket_to_index[sorted_buckets[i]]
        idx2 = bucket_to_index[sorted_buckets[i + 1]]
        edge_index.append([idx1, idx2])
        edge_index.append([idx2, idx1]) # Graph vô hướng

    if len(edge_index) == 0:
        edge_index.append([0, 0])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    x = torch.tensor(node_features, dtype=torch.float)
    y = torch.tensor(node_labels, dtype=torch.long)

    if x.size(1) != config.IN_FEATURES:
        raise ValueError(f"Feature dimension mismatch. Graph has {x.size(1)} features, config expects {config.IN_FEATURES}")

    graph_data = Data(x=x, edge_index=edge_index, y=y)

    # === THÊM PHẦN TÍNH TOÁN CẤU TRÚC (CHO CÔNG THỨC 8) ===
    # Thêm self-loops (nút tự kết nối với chính nó)
    edge_index_with_loops, _ = add_remaining_self_loops(graph_data.edge_index, num_nodes=graph_data.num_nodes)

    # Tính toán hệ số GCN (đại diện cho cấu trúc S_ij)
    # S_ij = 1 / sqrt(deg(i) * deg(j))
    _, gcn_coeffs = get_laplacian(edge_index_with_loops, normalization='sym', num_nodes=graph_data.num_nodes)

    # Lưu các hệ số này làm thuộc tính cạnh (edge attribute)
    graph_data.structure_coeffs = gcn_coeffs
    graph_data.edge_index = edge_index_with_loops # Sử dụng edge_index mới
    # === KẾT THÚC PHẦN THÊM ===

    print(f"Built graph with {x.size(0)} nodes and {graph_data.edge_index.size(1)} edges (with self-loops)")
    return graph_data

def prepare_federated_data(graph_data, num_clients=10):
    """(Sửa lỗi rò rỉ, bỏ test_mask)"""
    num_nodes = graph_data.num_nodes

    indices = torch.randperm(num_nodes)
    train_size = int(0.8 * num_nodes) # 80% train

    graph_data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    graph_data.val_mask = torch.zeros(num_nodes, dtype=torch.bool)

    graph_data.train_mask[indices[:train_size]] = True
    graph_data.val_mask[indices[train_size:]] = True # 20% val

    client_data = []
    nodes_per_client = num_nodes // num_clients

    for i in range(num_clients):
        start_idx = i * nodes_per_client
        end_idx = (i + 1) * nodes_per_client if i < num_clients - 1 else num_nodes

        # Client graph sẽ kế thừa tất cả thuộc tính (bao gồm structure_coeffs)
        client_graph = Data(
            x=graph_data.x.clone(),
            edge_index=graph_data.edge_index.clone(),
            y=graph_data.y.clone(),
            train_mask=graph_data.train_mask.clone(),
            val_mask=graph_data.val_mask.clone(),
            structure_coeffs=graph_data.structure_coeffs.clone() # Thêm dòng này
        )

        client_mask = torch.zeros(num_nodes, dtype=torch.bool)
        client_mask[start_idx:end_idx] = True
        client_graph.client_mask = client_mask

        client_data.append(client_graph)

    return client_data


def load_and_preprocess_datasets():
    """Main function to load and preprocess all datasets"""
    print("Loading NSL-KDD datasets...")
    # ... (code load và kiểm tra file) ...

    train_df = load_nsl_kdd_data(config.TRAIN_20P_FILE)
    test_df = load_nsl_kdd_data(config.TEST_FILE)

    if train_df is None or test_df is None:
        raise FileNotFoundError("Could not load dataset files")

    print("Preprocessing data...")
    train_processed, le_dict, scaler = preprocess_train_data(train_df)
    test_processed = preprocess_test_data(test_df.copy(), le_dict, scaler)
    # Cân bằng dữ liệu train
    train_processed = balance_dataset(train_processed)

    print("Building temporal graphs...")
    # Đảm bảo dùng config.TIME_WINDOW
    train_graph = build_temporal_graph(train_processed, config.TIME_WINDOW)
    test_graph = build_temporal_graph(test_processed, config.TIME_WINDOW)

    # === TÍNH TRỌNG SỐ LỚP  ===
    print("Calculating class weights...")
    labels = train_graph.y.numpy()
    counts = np.bincount(labels, minlength=config.OUT_FEATURES)
    weights = 1.0 / (counts + 1e-6) # +1e-6 để tránh chia cho 0
    #
    weights = np.sqrt(weights)

    weights = weights / np.mean(weights) # Chuẩn hóa
    weights_tensor = torch.tensor(weights, dtype=torch.float)
    print(f"Class weights (0-4): {weights_tensor.numpy()}")
    # === KẾT THÚC TÍNH TRỌNG SỐ ===

    os.makedirs(config.DATA_PROCESSED_PATH, exist_ok=True)
    torch.save(train_graph, os.path.join(config.DATA_PROCESSED_PATH, 'train_graph.pt'))
    torch.save(test_graph, os.path.join(config.DATA_PROCESSED_PATH, 'test_graph.pt'))

    print("Graphs saved to processed directory")
    return train_graph, test_graph, weights_tensor

Writing data_loader.py


In [7]:
%%writefile utils.py
import torch
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os
from config import config

# (Giữ nguyên các hàm: set_seed, save_training_results, load_training_results, plot_training_results)
def set_seed(seed=42):
    """Set random seed for reproducibility"""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

def save_training_results(results, filename='training_results.json'):
    """Save training results to JSON file"""
    os.makedirs(config.RESULTS_PATH, exist_ok=True)
    filepath = os.path.join(config.RESULTS_PATH, filename)

    serializable_results = {}
    for key, value in results.items():
        if isinstance(value, list) and len(value) > 0 and torch.is_tensor(value[0]):
            serializable_results[key] = [v.item() if torch.is_tensor(v) else v for v in value]
        elif isinstance(value, list) and len(value) > 0 and isinstance(value[0], (np.float32, np.float64)):
            serializable_results[key] = [float(v) for v in value]
        else:
            serializable_results[key] = value

    with open(filepath, 'w') as f:
        json.dump(serializable_results, f, indent=4)

    print(f"Results saved to {filepath}")

def load_training_results(filename='training_results.json'):
    """Load training results from JSON file"""
    filepath = os.path.join(config.RESULTS_PATH, filename)
    with open(filepath, 'r') as f:
        return json.load(f)

def plot_training_results(results, save_plot=True):
    """Plot training results"""
    fig, ((ax1, ax2)) = plt.subplots(1, 2, figsize=(15, 5))

    rounds = results.get('round', [])
    if not rounds:
        print("Không tìm thấy 'round' trong results để vẽ biểu đồ.")
        return

    ax1.plot(rounds, results.get('train_loss', [0]*len(rounds)), 'b-', label='Training Loss', linewidth=2)
    ax1.set_xlabel('Communication Rounds')
    ax1.set_ylabel('Loss')
    ax1.set_title('Federated Training Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(rounds, results.get('val_accuracy', [0]*len(rounds)), 'g-', label='Validation Accuracy', linewidth=2)
    ax2.plot(rounds, results.get('test_accuracy', [0]*len(rounds)), 'r-', label='Test Accuracy', linewidth=2)
    ax2.set_xlabel('Communication Rounds')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Federated Learning Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()

    if save_plot:
        os.makedirs(config.RESULTS_PATH, exist_ok=True)
        plt.savefig(os.path.join(config.RESULTS_PATH, 'training_plots.png'), dpi=300, bbox_inches='tight')

    try:
        plt.show()
    except Exception as e:
        print(f"Không thể hiển thị biểu đồ (có thể đang chạy trên server không có GUI): {e}")

# === CẬP NHẬT HÀM evaluate_model ===
def evaluate_model(server, test_graph, save_results=True):
    """Comprehensive model evaluation for 5-class classification"""
    server.global_model.eval()
    server.global_model.to(config.DEVICE)

    # Chuyển test_graph sang device
    test_graph = test_graph.to(config.DEVICE)

    with torch.no_grad():
        # === SỬA LẠI HÀM FORWARD ===
        out = server.global_model(
            test_graph.x,
            test_graph.edge_index,
            test_graph.structure_coeffs # Truyền thêm
        )
        pred = out.argmax(dim=1)

        y_true = test_graph.y.cpu().numpy()
        y_pred = pred.cpu().numpy()

        accuracy = accuracy_score(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred)

        # Cập nhật target names cho 5 classes
        target_names = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']
        report = classification_report(y_true, y_pred, target_names=target_names, output_dict=True, zero_division=0)

        print("=== Final Model Evaluation (5-class) ===")
        print(f"Accuracy: {accuracy:.4f}")
        print("\nConfusion Matrix:")
        print(cm)
        print("\nClassification Report:")
        print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

        evaluation_metrics = {
            'accuracy': accuracy,
            'confusion_matrix': cm.tolist(),
            'classification_report': report
        }

        if save_results:
            save_training_results(evaluation_metrics, 'evaluation_metrics.json')

        return evaluation_metrics

def print_system_info():
    """Print system information"""
    print("=== System Information ===")
    print(f"Device: {config.DEVICE}")
    print(f"PyTorch version: {torch.__version__}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name()}")
        print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of clients: {config.NUM_CLIENTS}")
    print(f"Communication rounds: {config.COMMUNICATION_ROUNDS}")
    print(f"Local epochs: {config.LOCAL_EPOCHS}")
    print(f"TIME_WINDOW: {config.TIME_WINDOW}")
    print("==========================")


Writing utils.py


In [8]:
%%writefile federated.py
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler # Import AMP
from model import GATWithAttention
from config import config
import gc

class FedGATClient:
    def __init__(self, client_id, data):
        self.client_id = client_id
        self.data = data
        self.model = GATWithAttention(
            config.IN_FEATURES,
            config.HIDDEN_FEATURES,
            config.OUT_FEATURES,
            heads=config.HEADS,
            dropout=config.DROPOUT
        ).to(config.DEVICE)

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )
        self.class_weights = config.CLASS_WEIGHTS
        self.scaler = GradScaler() # Khởi tạo Scaler cho AMP

    def train_epoch(self):
        self.model.train()
        self.optimizer.zero_grad()

        train_mask = self.data.train_mask & self.data.client_mask
        if train_mask.sum() == 0:
            return 0.0, self.model.state_dict()

        # === DÙNG MIXED PRECISION (AMP) ĐỂ GIẢM MEMORY ===
        try:
            with autocast():
                out = self.model(
                    self.data.x,
                    self.data.edge_index,
                    self.data.structure_coeffs
                )
                loss = F.nll_loss(
                    out[train_mask],
                    self.data.y[train_mask],
                    weight=self.class_weights
                )

            # Backward với Scaler
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            # Dọn dẹp bộ nhớ cache ngay sau mỗi epoch
            torch.cuda.empty_cache()
            return loss.item(), self.model.state_dict()

        except torch.OutOfMemoryError:
            print(f"Client {self.client_id}: OOM skipped.")
            torch.cuda.empty_cache()
            return 0.0, self.model.state_dict()

    def evaluate(self):
        self.model.eval()
        mask = self.data.val_mask & self.data.client_mask
        if mask.sum() == 0: return 0.0

        with torch.no_grad():
            out = self.model(
                self.data.x,
                self.data.edge_index,
                self.data.structure_coeffs
            )
            pred = out.argmax(dim=1)
            correct = (pred[mask] == self.data.y[mask]).sum()
            acc = int(correct) / int(mask.sum())
        return acc

class FedGATServer:
    def __init__(self):
        self.global_model = GATWithAttention(
            config.IN_FEATURES,
            config.HIDDEN_FEATURES,
            config.OUT_FEATURES,
            heads=config.HEADS,
            dropout=config.DROPOUT
        )

    def aggregate(self, client_updates, client_data_sizes):
        total_data_size = sum(client_data_sizes)
        if total_data_size == 0: return

        global_state_dict = self.global_model.state_dict()
        # Reset về 0
        for key in global_state_dict.keys():
            global_state_dict[key] = torch.zeros_like(global_state_dict[key], device='cpu')

        # Cộng dồn (trên CPU để tiết kiệm GPU VRAM)
        for i, client_state_dict in enumerate(client_updates):
            weight = client_data_sizes[i] / total_data_size
            for key in global_state_dict.keys():
                global_state_dict[key] += client_state_dict[key].cpu() * weight

        self.global_model.load_state_dict(global_state_dict)

    def distribute_model(self, clients):
        global_state_dict = self.global_model.state_dict()
        for client in clients:
            client.model.load_state_dict(global_state_dict)

    def save_model(self, path):
        torch.save(self.global_model.cpu().state_dict(), path)

    def load_model(self, path):
        self.global_model.load_state_dict(torch.load(path, map_location=config.DEVICE))

Writing federated.py


In [9]:
%%writefile train.py
import os
import torch
import sys
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

try:
    from data_loader import load_and_preprocess_datasets, prepare_federated_data
    from federated import FedGATClient, FedGATServer
    from utils import set_seed, evaluate_model, print_system_info
    from config import config
except ImportError:
    sys.path.append(os.path.dirname(os.path.abspath(__file__)))
    from data_loader import load_and_preprocess_datasets, prepare_federated_data
    from federated import FedGATClient, FedGATServer
    from utils import set_seed, evaluate_model, print_system_info
    from config import config

def train_fedgat():
    set_seed(config.SEED)
    print_system_info()

    os.makedirs(config.MODELS_PATH, exist_ok=True)
    os.makedirs(config.RESULTS_PATH, exist_ok=True)

    # 1. Load Data
    print(f"Loading {config.TRAIN_20P_FILE}...")
    train_graph, test_graph, class_weights = load_and_preprocess_datasets()
    config.CLASS_WEIGHTS = class_weights.to(config.DEVICE)
    test_graph = test_graph.to(config.DEVICE)

    # 2. Setup Federated Learning
    print("Distributing data to clients...")
    client_data_list = prepare_federated_data(train_graph, num_clients=config.NUM_CLIENTS)
    client_data_list = [data.to(config.DEVICE) for data in client_data_list]

    server = FedGATServer()
    clients = [FedGATClient(i, data) for i, data in enumerate(client_data_list)]
    server.global_model.to(config.DEVICE)

    results = {'round': [], 'loss': [], 'val_acc': [], 'test_acc': [], 'test_f1': []}
    best_val_acc = 0.0
    patience_counter = 0

    print(f"Starting training for {config.COMMUNICATION_ROUNDS} rounds...")

    for round_idx in range(config.COMMUNICATION_ROUNDS):
        print(f"\n--- Round {round_idx + 1} ---")

        client_updates = []
        client_data_sizes = []
        total_train_loss = 0
        total_val_acc = 0
        active_clients = 0

        # --- HUẤN LUYỆN & VALIDATION TRÊN CLIENT ---
        for client in clients:
            # Train
            round_loss = 0
            for epoch in range(config.LOCAL_EPOCHS):
                loss, client_state_dict = client.train_epoch()
                round_loss += loss

            # Validation (Quan trọng: Đánh giá trên tập Val riêng của Client)
            val_acc = client.evaluate()

            data_size = (client.data.train_mask & client.data.client_mask).sum().item()
            if data_size > 0:
                client_updates.append(client_state_dict)
                client_data_sizes.append(data_size)
                total_train_loss += round_loss / config.LOCAL_EPOCHS
                total_val_acc += val_acc
                active_clients += 1

        # --- TỔNG HỢP & ĐÁNH GIÁ TRÊN SERVER ---
        if active_clients > 0:
            # 1. Aggregate
            server.aggregate(client_updates, client_data_sizes)
            server.distribute_model(clients)

            # 2. Calculate Average Metrics
            avg_loss = total_train_loss / active_clients
            avg_val_acc = total_val_acc / active_clients # Trung bình Val Acc của các Client

            # 3. Test on Server (Global Test Set)
            server.global_model.eval()
            with torch.no_grad():
                out = server.global_model(test_graph.x, test_graph.edge_index, test_graph.structure_coeffs)
                pred = out.argmax(dim=1)
                y_true = test_graph.y.cpu().numpy()
                y_pred = pred.cpu().numpy()

                test_acc = (y_pred == y_true).sum() / len(y_true)
                prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

            # 4. Print & Save
            print(f"Train Loss: {avg_loss:.4f}")
            print(f"Val Acc:    {avg_val_acc:.4f} (Avg of {active_clients} clients)")
            print(f"Test Acc:   {test_acc:.4f}")
            print(f"Macro F1:   {f1:.4f} | Recall: {rec:.4f}")

            results['round'].append(round_idx + 1)
            results['loss'].append(avg_loss)
            results['val_acc'].append(avg_val_acc)
            results['test_acc'].append(test_acc)
            results['test_f1'].append(f1)

            # 5. Checkpoint & Early Stopping
            if avg_val_acc > best_val_acc:
                best_val_acc = avg_val_acc
                patience_counter = 0
                torch.save(server.global_model.state_dict(), os.path.join(config.MODELS_PATH, 'best_model.pth'))
                print("-> New best model saved!")
            else:
                patience_counter += 1

            if patience_counter >= 15: # Dừng nếu 15 vòng không cải thiện Val Acc
                print("Early stopping triggered!")
                break
        else:
            print("No active clients.")

    # Kết thúc
    print("\nTraining finished.")
    # Load lại model tốt nhất để đánh giá cuối cùng
    server.global_model.load_state_dict(torch.load(os.path.join(config.MODELS_PATH, 'best_model.pth')))
    evaluate_model(server, test_graph.cpu())

if __name__ == "__main__":
    train_fedgat()

Writing train.py


In [10]:
import os
import torch

# Cấu hình quản lý bộ nhớ của PyTorch để giảm phân mảnh
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Xóa cache bộ nhớ cũ
torch.cuda.empty_cache()
import gc
gc.collect()
print("Đã giải phóng bộ nhớ và cấu hình lại môi trường.")

Đã giải phóng bộ nhớ và cấu hình lại môi trường.


In [11]:
!python train.py

=== System Information ===
Device: cuda
PyTorch version: 2.8.0+cu126
GPU: Tesla T4
CUDA version: 12.6
Number of clients: 10
Communication rounds: 50
Local epochs: 10
TIME_WINDOW: 1
Loading KDDTrain+_20Percent.txt...
Loading NSL-KDD datasets...
File not found: /data/raw/KDDTrain+_20Percent.txt
Found file at alternative path: data/raw/KDDTrain+_20Percent.txt
Successfully loaded KDDTrain+_20Percent.txt with 25192 records
File not found: /data/raw/KDDTest+.txt
Found file at alternative path: data/raw/KDDTest+.txt
Successfully loaded KDDTest+.txt with 22544 records
Preprocessing data...
Balancing dataset (Oversampling rare classes)...
Class distribution after balancing:
label
0    13449
1     9234
3     4617
2     4617
4     2308
Name: count, dtype: int64
Building temporal graphs...
Built graph with 34225 nodes and 102673 edges (with self-loops)
Built graph with 22544 nodes and 67630 edges (with self-loops)
Calculating class weights...
Class weights (0-4): [0.6223359 0.7510605 1.0621599 1.0